# Experiment 05 — Embedding-space Visualisation (UMAP + PCA + Silhouette)
**Purpose.**  RYAI Reviewer 3 (#9) and Meta-Rad Reviewer 2 (#3) both ask
whether clean and perturbed embeddings *separate* in feature space.
This experiment produces:
 * UMAP projection of clean vs perturbed CLS embeddings per model
 * Silhouette score quantifying separation
 * Inter-cluster Mahalanobis distance (dimension-normalised)
 * PCA top-2 as a simple baseline visual
**Inputs.**  Uses embeddings produced by exp04 (clean + perturbed CLS).
**How to run.**
    export DATASET=nih                 # or 'emory'
    jupyter nbconvert --execute --to notebook 05_EmbeddingVisualization_UMAP.ipynb
Output: `exp05_<dataset>_embedding_separation.parquet` + PNGs.

In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory
MODELS = "raddino,dinov2,dinov3,biomedclip,medsiglip"
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None only when running gated models locally


In [ ]:
# Apply papermill parameters to environment (no-op when env vars already set)
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("MODELS_TO_RUN", MODELS)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [1]:
import os, sys, gc, json
from pathlib import Path
REPO_ROOT = Path(os.environ.get("REPO_ROOT", "/home/saptpurk/embeddings-noise-eliminators"))
sys.path.insert(0, str(REPO_ROOT))

from common import get_config, PARAMS, MODELS

CFG = get_config()
EXP04_DIR = CFG.output_dir("exp04_clean_vs_perturbed")
OUT       = CFG.output_dir("exp05_embedding_viz")
print(f"Dataset: {CFG.name}  |  Reading from: {EXP04_DIR}  |  Writing: {OUT}")

Dataset: MIMIC-CXR-JPG  |  Reading from: /home/saptpurk/embeddings-noise-eliminators/v4_work/v4_exp04_clean_vs_perturbed_mimic  |  Writing: /home/saptpurk/embeddings-noise-eliminators/v4_work/v4_exp05_embedding_viz_mimic


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn not installed; falling back to PCA only.")
    print("    pip install umap-learn")

from common import models_to_run
MODEL_NAMES   = models_to_run()
print(f"Running models: {MODEL_NAMES}")
PERTURBATIONS = [
    "iso_blur_p4", "iso_blur_p8",
    "dir_blur_cranio_p32", "dir_blur_cranio_p64",
    "dir_blur_lateral_p32", "dir_blur_lateral_p64",
    "reticular_fine_p32", "reticular_coarse_p64",
    "ground_glass_p64",
]

Running models: ['dinov3', 'medsiglip']


In [3]:
def mahalanobis_between(A, B):
    """
    Robust inter-cluster Mahalanobis distance using pooled covariance shrinkage.
    """
    mu_a, mu_b = A.mean(0), B.mean(0)
    pooled = 0.5 * (np.cov(A.T) + np.cov(B.T))
    # Shrinkage to avoid singularity in high-D
    trace = np.trace(pooled) / pooled.shape[0]
    pooled = 0.9 * pooled + 0.1 * trace * np.eye(pooled.shape[0])
    try:
        inv = np.linalg.pinv(pooled)
        d = (mu_a - mu_b) @ inv @ (mu_a - mu_b).T
        return float(np.sqrt(max(d, 0.0)))
    except np.linalg.LinAlgError:
        return float(np.linalg.norm(mu_a - mu_b))


def silhouette_binary(X, y, subsample=5000, seed=42):
    """Binary silhouette on a subsample (full silhouette is O(n^2) memory)."""
    rng = np.random.default_rng(seed)
    n = len(X)
    if n > subsample:
        idx = rng.choice(n, size=subsample, replace=False)
        X, y = X[idx], y[idx]
    try:
        return float(silhouette_score(X, y, metric="cosine"))
    except Exception:
        return float("nan")

In [4]:
records = []
for model_name in MODEL_NAMES:
    clean_path = EXP04_DIR / f"emb_clean_{model_name}_test.npy"
    if not clean_path.exists():
        print(f"Skipping {model_name}: exp04 outputs not found.")
        continue
    X_clean = np.load(clean_path)

    for pert in PERTURBATIONS:
        pert_path = EXP04_DIR / f"emb_pert_{model_name}_{pert}_test.npy"
        if not pert_path.exists():
            continue
        X_pert = np.load(pert_path)

        # Stack with labels
        X = np.vstack([X_clean, X_pert]).astype(np.float32)
        y = np.concatenate([np.zeros(len(X_clean)), np.ones(len(X_pert))]).astype(int)

        # Metrics
        sil = silhouette_binary(X, y)
        maha = mahalanobis_between(X_clean, X_pert)

        # PCA 2-D
        pca = PCA(n_components=2, random_state=PARAMS.random_seed)
        Z_pca = pca.fit_transform(X)

        # UMAP 2-D (sub-sample for speed)
        if HAS_UMAP:
            rng = np.random.default_rng(PARAMS.random_seed)
            max_plot = min(10000, len(X))
            idx = rng.choice(len(X), size=max_plot, replace=False)
            reducer = umap.UMAP(n_components=2, n_neighbors=30,
                                min_dist=0.1, metric="cosine",
                                random_state=PARAMS.random_seed)
            Z_umap = reducer.fit_transform(X[idx])
            y_plot = y[idx]
        else:
            Z_umap, y_plot = None, None

        # Plot
        fig, axes = plt.subplots(1, 2 if HAS_UMAP else 1,
                                 figsize=(10 if HAS_UMAP else 5, 4))
        if not HAS_UMAP: axes = [axes]
        axes[0].scatter(Z_pca[y == 0, 0], Z_pca[y == 0, 1],
                        s=3, alpha=0.5, label="clean")
        axes[0].scatter(Z_pca[y == 1, 0], Z_pca[y == 1, 1],
                        s=3, alpha=0.5, label="perturbed", c="crimson")
        axes[0].set_title(f"PCA | {model_name} | {pert}")
        axes[0].legend()
        if HAS_UMAP:
            axes[1].scatter(Z_umap[y_plot == 0, 0], Z_umap[y_plot == 0, 1],
                            s=3, alpha=0.5, label="clean")
            axes[1].scatter(Z_umap[y_plot == 1, 0], Z_umap[y_plot == 1, 1],
                            s=3, alpha=0.5, label="perturbed", c="crimson")
            axes[1].set_title(f"UMAP | silhouette={sil:.3f}")
            axes[1].legend()
        plt.tight_layout()
        png = OUT / f"viz_{model_name}_{pert}.png"
        plt.savefig(png, dpi=180, bbox_inches="tight")
        plt.close()

        records.append(dict(
            dataset=CFG.dataset, model=model_name, perturbation=pert,
            silhouette=sil, mahalanobis=maha,
            n_clean=int(len(X_clean)), n_perturbed=int(len(X_pert)),
            plot_path=str(png.relative_to(OUT)),
        ))
        print(f"{model_name:>8s} {pert:>22s}  silhouette={sil:+.3f}  mahalanobis={maha:.3f}")

/home/saptpurk/embeddings-noise-eliminators/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  dinov3            iso_blur_p4  silhouette=-0.000  mahalanobis=0.014


/home/saptpurk/embeddings-noise-eliminators/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  dinov3            iso_blur_p8  silhouette=-0.000  mahalanobis=0.021


/home/saptpurk/embeddings-noise-eliminators/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

medsiglip     reticular_fine_p32  silhouette=-0.000  mahalanobis=0.044


/home/saptpurk/embeddings-noise-eliminators/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


medsiglip   reticular_coarse_p64  silhouette=+0.007  mahalanobis=1.470


/home/saptpurk/embeddings-noise-eliminators/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


medsiglip       ground_glass_p64  silhouette=-0.000  mahalanobis=0.185


In [5]:
df = pd.DataFrame(records)
_run_tag = os.environ.get("RUN_TAG", "")
_suffix = ("_" + _run_tag) if _run_tag else ""
out_path = OUT / f"exp05_{CFG.dataset}_embedding_separation{_suffix}.parquet"
df.to_parquet(out_path, index=False)
print(f"\nSaved -> {out_path}")
df


Saved -> /home/saptpurk/embeddings-noise-eliminators/v4_work/v4_exp05_embedding_viz_mimic/exp05_mimic_embedding_separation_gpu1.parquet


,dataset,model,perturbation,silhouette,mahalanobis,n_clean,n_perturbed,plot_path
0,mimic,dinov3,iso_blur_p4,-0.000035,0.013773,11072,11072,viz_dinov3_iso_blur_p4.png
1,mimic,dinov3,iso_blur_p8,-0.000036,0.020902,11072,11072,viz_dinov3_iso_blur_p8.png
2,mimic,dinov3,dir_blur_cranio_p32,-0.000031,0.120051,11072,11072,viz_dinov3_dir_blur_cranio_p32.png
3,mimic,dinov3,dir_blur_cranio_p64,0.000088,0.774521,11072,11072,viz_dinov3_dir_blur_cranio_p64.png
4,mimic,dinov3,dir_blur_lateral_p32,-0.000033,0.110737,11072,11072,viz_dinov3_dir_blur_lateral_p32.png
5,mimic,dinov3,dir_blur_lateral_p64,0.000069,0.718655,11072,11072,viz_dinov3_dir_blur_lateral_p64.png
6,mimic,dinov3,reticular_fine_p32,-0.000032,0.069454,11072,11072,viz_dinov3_reticular_fine_p32.png
7,mimic,dinov3,reticular_coarse_p64,0.000001,0.288042,11072,11072,viz_dinov3_reticular_coarse_p64.png
8,mimic,dinov3,ground_glass_p64,-0.000031,0.241097,11072,11072,viz_dinov3_ground_glass_p64.png
9,mimic,medsiglip,iso_blur_p4,-0.000122,0.006911,11072,11072,viz_medsiglip_iso_blur_p4.png
